# Customers Data Cleaning

Problems to investigate
- Unique customer ids (i.e unique transactions)
- Do all customer ids correlate to a valid/existing order in the orders dataset
- Customer zipcode, city and states are valid
- Any white space

In [2]:
import pandas as pd 

customers_raw = pd.read_csv('../data/raw/olist_customers_dataset.csv')
customers = customers_raw.copy()

print("Customers shape:", customers_raw.shape)
print("Working customers shape:", customers.shape)

Customers shape: (99441, 5)
Working customers shape: (99441, 5)


In [3]:
# Ensure all customer ids are unique

print("Duplicate customer ids:", customers["customer_id"].duplicated().sum())

Duplicate customer ids: 0


In [4]:
# Do all customer ids correlate to an existing order in orders dataset

orders = pd.read_csv('../data/processed/olist_orders_cleaned.csv')

customer_order_check = customers.merge(
    orders[["order_id", "customer_id", "order_status"]],
    on="customer_id",
    how="left",
    indicator=True
)

print(customer_order_check["_merge"].value_counts())

_merge
both          99441
left_only         0
right_only        0
Name: count, dtype: int64


In [5]:
# Ensure all zipcodes, city and states are valid
import unicodedata

geo = pd.read_csv('../data/processed/olist_geolocation_cleaned.csv')

def normalize_city(city):
    return "".join(
        character for character in unicodedata.normalize("NFD", city)
        if unicodedata.category(character) != "Mn"
    ).lower().strip()

customers["city_normalized"] = customers["customer_city"].apply(normalize_city)

geo["city_normalized"] = geo["geolocation_city"].apply(normalize_city)

city_check = customers.merge(
    geo[
        [
            "geolocation_zip_code_prefix",
            "city_normalized",
            "geolocation_state"
        ]
    ].drop_duplicates(),
    left_on=[
        "customer_zip_code_prefix",
        "city_normalized",
        "customer_state"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "city_normalized",
        "geolocation_state"
    ],
    how="left",
    indicator=True
)

print(city_check["_merge"].value_counts())

_merge
both          99139
left_only       302
right_only        0
Name: count, dtype: int64


In [6]:
# Investigate unmatched customer locs
unmatched_cities = city_check[
    city_check["_merge"] == "left_only"
]

print("Unmatched customer records:", len(unmatched_cities))

display(
    unmatched_cities[
        [
            "customer_zip_code_prefix",
            "customer_city",
            "city_normalized",
            "customer_state"
        ]
    ].drop_duplicates().sort_values("customer_zip_code_prefix")
)

Unmatched customer records: 302


,customer_zip_code_prefix,customer_city,city_normalized,customer_state
60170,2140,sao paulo,sao paulo,SP
3064,6930,cipo-guacu,cipo-guacu,SP
73328,7412,aruja,aruja,SP
32512,7430,aruja,aruja,SP
1272,7729,caieiras,caieiras,SP
...,...,...,...,...
56640,87511,umuarama,umuarama,PR
73258,93602,estancia velha,estancia velha,RS
69274,94370,ipiranga,ipiranga,RS
24146,95572,mampituba,mampituba,RS


In [7]:
# Check whether the ZIP prefix and state exist in Geolocation
zip_state_check = unmatched_cities.merge(
    geo[
        [
            "geolocation_zip_code_prefix",
            "geolocation_state"
        ]
    ].drop_duplicates(),
    left_on=[
        "customer_zip_code_prefix",
        "customer_state"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_state"
    ],
    how="left",
    indicator="zip_state_match"
)

print(zip_state_check["zip_state_match"].value_counts())

zip_state_match
left_only     278
both           24
right_only      0
Name: count, dtype: int64


In [8]:
# Add data quality flag

customers['missing_geo'] = (city_check["_merge"] == "left_only").values

print(customers["missing_geo"].value_counts())

missing_geo
False    99139
True       302
Name: count, dtype: int64


In [9]:
# Check for any whitespace
string_columns = customers.select_dtypes(
    include=["object", "str"]
).columns

for column in string_columns:
    whitespace_count = customers[column].str.contains(
        r"^\s|\s$", na=False
    ).sum()

    print(f"{column}: {whitespace_count} values with whitespace")

for column in string_columns:
    print(
        f"{column}:",
        (customers[column] == "").sum(),
        "empty strings"
    )

customers.columns = (
    customers.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(customers.columns.tolist())


customer_id: 0 values with whitespace
customer_unique_id: 0 values with whitespace
customer_city: 0 values with whitespace
customer_state: 0 values with whitespace
city_normalized: 0 values with whitespace
customer_id: 0 empty strings
customer_unique_id: 0 empty strings
customer_city: 0 empty strings
customer_state: 0 empty strings
city_normalized: 0 empty strings
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'city_normalized', 'missing_geo']


In [10]:
customers.to_csv(
    "../data/processed/olist_customers_cleaned.csv",
    index=False
)